# Quality of Care Indicators (SNT)

Indicateurs de qualité des soins à partir des données routine DHIS2 (fichier imputé).

**Indicateurs:** Testing rate (TEST/SUSP), Treatment rate (MALTREAT/CONF), Case fatality (MALDTH/MALADM), Prop. admissions paludisme (MALADM/ALLADM), Prop. décès paludisme (MALDTH/ALLDTH), PRES et ALLOUT en effectifs. Agrégation par district et année.

In [ ]:
# Parameters (injected by pipeline when run via OpenHexa)
import os
if "ROOT_PATH" not in dir() and not os.environ.get("ROOT_PATH"):
    ROOT_PATH = "/home/hexa/workspace"  # default in OpenHexa
else:
    ROOT_PATH = os.environ.get("ROOT_PATH", globals().get("ROOT_PATH", "/home/hexa/workspace"))
if "OUTLIER_IMPUTATION_METHOD" not in dir() and not os.environ.get("OUTLIER_IMPUTATION_METHOD"):
    OUTLIER_IMPUTATION_METHOD = "mean"
else:
    OUTLIER_IMPUTATION_METHOD = os.environ.get("OUTLIER_IMPUTATION_METHOD", globals().get("OUTLIER_IMPUTATION_METHOD", "mean"))

## 1. Setup & config

In [ ]:
import json
from pathlib import Path

root = Path(ROOT_PATH)
config_path = root / "configuration" / "SNT_config.json"
with open(config_path, encoding="utf-8") as f:
    config = json.load(f)

COUNTRY_CODE = config["SNT_CONFIG"]["COUNTRY_CODE"]
DATA_PATH = root / "data" / "dhis2"
OUTLIERS_PATH = DATA_PATH / "outliers_imputation"
OUT_PATH = DATA_PATH / "quality_of_care"
REPORTING_OUTPUTS = root / "pipelines" / "snt_quality_of_care" / "reporting" / "outputs"

OUT_PATH.mkdir(parents=True, exist_ok=True)
REPORTING_OUTPUTS.mkdir(parents=True, exist_ok=True)

method = OUTLIER_IMPUTATION_METHOD.lower()
if method == "trend":
    routine_file = OUTLIERS_PATH / f"{COUNTRY_CODE}_routine_outliers-trend_imputed.parquet"
else:
    routine_file = OUTLIERS_PATH / f"{COUNTRY_CODE}_routine_outliers-{method}_imputed.parquet"

print("COUNTRY_CODE:", COUNTRY_CODE)
print("Routine file:", routine_file, "exists:", routine_file.exists())

In [ ]:
import pandas as pd
import numpy as np

if not routine_file.exists():
    raise FileNotFoundError(f"Routine imputed file not found: {routine_file}. Run the outliers imputation pipeline first.")

routine = pd.read_parquet(routine_file)
cols_num = ["TEST", "SUSP", "MALTREAT", "CONF", "MALDTH", "MALADM", "ALLADM", "ALLDTH", "PRES", "ALLOUT"]
for c in cols_num:
    if c in routine.columns:
        routine[c] = pd.to_numeric(routine[c], errors="coerce").fillna(0)

if "YEAR" not in routine.columns and "MONTH" in routine.columns:
    routine["YEAR"] = pd.to_numeric(routine["MONTH"].astype(str).str[:4], errors="coerce")
routine["YEAR"] = pd.to_numeric(routine["YEAR"], errors="coerce")
routine = routine.dropna(subset=["YEAR"])
print(routine.shape)
print(routine.columns.tolist())

## 2. Aggregate by district and year

In [ ]:
id_cols = ["ADM2_ID", "YEAR"]
if "ADM2_NAME" in routine.columns:
    id_cols = ["ADM2_ID", "ADM2_NAME", "YEAR"]

agg_cols = [c for c in cols_num if c in routine.columns]
agg = routine.groupby(id_cols, as_index=False)[agg_cols].sum()

agg["testing_rate"] = np.where(agg["SUSP"] > 0, agg["TEST"] / agg["SUSP"], np.nan)
agg["treatment_rate"] = np.where(agg["CONF"] > 0, agg["MALTREAT"] / agg["CONF"], np.nan)
agg["case_fatality_rate"] = np.where(agg["MALADM"] > 0, agg["MALDTH"] / agg["MALADM"], np.nan)
agg["prop_adm_malaria"] = np.where(agg["ALLADM"] > 0, agg["MALADM"] / agg["ALLADM"], np.nan)
agg["prop_malaria_deaths"] = np.where(agg["ALLDTH"] > 0, agg["MALDTH"] / agg["ALLDTH"], np.nan)

map_df = agg.copy()
print(map_df.head())
print(map_df.shape)

## 3. Export data

In [ ]:
map_df.to_parquet(OUT_PATH / f"{COUNTRY_CODE}_quality_of_care.parquet", index=False)
map_df.to_csv(OUT_PATH / f"{COUNTRY_CODE}_quality_of_care.csv", index=False)

with pd.ExcelWriter(OUT_PATH / f"{COUNTRY_CODE}_quality_of_care.xlsx", engine="openpyxl") as writer:
    map_df.to_excel(writer, sheet_name="quality_of_care", index=False)

print("Exported parquet, csv, xlsx to", OUT_PATH)

## 4. Load shapes and plot maps

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import rgb2hex
from matplotlib.patches import Patch

shapes_path = None
for base in [root / "data" / "dhis2", root / "data", root]:
    for p in base.rglob("*shapes*.geojson"):
        if COUNTRY_CODE in p.name or "NER" in p.name or "shapes" in p.name:
            shapes_path = p
            break
    if shapes_path:
        break
if shapes_path is None:
    for p in (root / "data").rglob("*.geojson"):
        shapes_path = p
        break

if shapes_path is None or not shapes_path.exists():
    print("[WARNING] No shapes found; skipping maps.")
    gdf = None
else:
    gdf = gpd.read_file(shapes_path)
    if "ADM2_ID" not in gdf.columns and "id" in gdf.columns:
        gdf["ADM2_ID"] = gdf["id"]
    if "ADM2_NAME" not in gdf.columns and "name" in gdf.columns:
        gdf["ADM2_NAME"] = gdf["name"]
    print("Shapes:", len(gdf), gdf.columns.tolist())

In [ ]:
def _build_bins_labels_pct():
    bins = [0, 0.5, 0.75, 0.9, 1.0, 1.1, 100]
    labels = ["0-50%", "50-75%", "75-90%", "90-100%", "100%", ">100%", "N/D"]
    colors = ["#d73027", "#fc8d59", "#fee08b", "#d9ef8b", "#91cf60", "#1a9850", "#d3d3d3"]
    return bins, labels, colors

def plot_yearly_maps(gdf, map_df, value_col, title, is_pct=True, save_name="qoc_map"):
    if gdf is None or value_col not in map_df.columns:
        return
    bins, labels, color_steps = _build_bins_labels_pct()
    color_map = {lab: color_steps[i] for i, lab in enumerate(labels)}
    years = sorted(map_df["YEAR"].dropna().unique())
    for year in years:
        year_data = map_df[map_df["YEAR"] == year][["ADM2_ID", value_col]].drop_duplicates(subset="ADM2_ID")
        work = gdf.merge(year_data, on="ADM2_ID", how="left")
        if value_col not in work.columns:
            work[value_col] = np.nan
        vals = work[value_col]
        work["_cat"] = pd.cut(vals, bins=bins, labels=labels[: len(bins) - 1], include_lowest=True)
        work["_cat"] = work["_cat"].astype(object)
        work.loc[vals.isna(), "_cat"] = "N/D"
        work["_color"] = work["_cat"].map(color_map).astype(object)
        work.loc[work["_color"].isna(), "_color"] = "#d3d3d3"
        fig, ax = plt.subplots(1, 1, figsize=(10, 8))
        work.plot(ax=ax, color=work["_color"], edgecolor="#333", linewidth=0.3)
        ax.set_title(f"{title} — {int(year)}")
        ax.axis("off")
        patches = [Patch(facecolor=color_map[l], label=l) for l in labels]
        ax.legend(handles=patches, loc="upper left", bbox_to_anchor=(0.02, 0.98), frameon=True, fontsize=9)
        out_file = REPORTING_OUTPUTS / f"{save_name}_{int(year)}.png"
        fig.savefig(out_file, dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)
    return

In [ ]:
if gdf is not None:
    plot_yearly_maps(gdf, map_df, "testing_rate", "Testing rate (TEST / SUSP)", save_name="qoc_testing_rate")
    plot_yearly_maps(gdf, map_df, "treatment_rate", "Treatment rate (MALTREAT / CONF)", save_name="qoc_treatment_rate")
    plot_yearly_maps(gdf, map_df, "case_fatality_rate", "Case fatality rate (MALDTH / MALADM)", save_name="qoc_case_fatality")
    plot_yearly_maps(gdf, map_df, "prop_adm_malaria", "Prop. admissions paludisme (MALADM / ALLADM)", save_name="qoc_prop_adm_malaria")
    plot_yearly_maps(gdf, map_df, "prop_malaria_deaths", "Prop. décès paludisme (MALDTH / ALLDTH)", save_name="qoc_prop_malaria_deaths")
print("Maps saved to", REPORTING_OUTPUTS)